# [Chapter 8 — Parameter Estimation, §8.6] Seasonal Influenza Case Study: $\hat\alpha$ vs $\hat\lambda$ in Practice

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 8 — Parameter Estimation
**Considerations developed:** 8, 9, 13 (real-world applications)
**Estimated runtime:** ≤ 1 minute

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Applies the central comparison to a realistic seasonal-influenza scenario where prior immunity (~30% recovered from previous seasons) makes the susceptible-viewpoint estimator badly biased.

## What you should already know
Chapter 8 central comparison notebook.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt
from shared import (
    book_style,
    BOOK_COLORS,
    integrate_sir_i,
    baseline_central_comparison,
)
from shared.parameters import baseline_chapter_08
from shared.seeds import set_seed_chapter_08
from shared.verification import assert_within_tolerance

set_seed_chapter_08()
book_style()


## The scenario

Seasonal influenza occurs in a population where some individuals are already immune from prior infection or vaccination. A typical assumption: 30% of the population is *not* susceptible at season start.

This is exactly the regime where $\hat\lambda = J/N$ (assuming everyone susceptible) gives a biased estimate, while $\hat\alpha = J/I$ is robust.

In [ ]:
# Build a 'flu season' scenario
params = baseline_chapter_08()
prior_immunity_fraction = 0.30
params['S0'] = 1.0 - prior_immunity_fraction - 0.001
params['R0_init'] = prior_immunity_fraction
params['I0'] = 0.001

true_R0 = params['c_I'] * params['beta'] * params['tau_R']
result = integrate_sir_i(params)
t, S, I = result['t'], result['S'], result['I']

alpha_true = params['c_I'] * params['beta'] * (S / params['N'])
J_true = alpha_true * I
J_obs = np.maximum(J_true + 0.05 * J_true.max() * np.random.randn(len(t)), 0)

# Naive lambda_hat (assumes S* = N, ignoring prior immunity)
mask = (t > 3) & (t < 18)
S_star_naive = params['N']
lambda_hat_naive = J_obs[mask].mean() / S_star_naive
R0_lambda_naive = lambda_hat_naive * params['tau_R'] / (I[mask].mean() / params['N'])

# Sophisticated lambda_hat (uses prior immunity correction)
S_star_correct = (1 - prior_immunity_fraction) * params['N']
lambda_hat_correct = J_obs[mask].mean() / S_star_correct
R0_lambda_correct = lambda_hat_correct * params['tau_R'] / (I[mask].mean() / params['N'])

# alpha_hat (uses observed I)
alpha_hat = (J_obs[mask] / I[mask]).mean()
S_avg = S[mask].mean() / params['N']
R0_alpha = alpha_hat * params['tau_R'] / S_avg

print(f"True R_0 (the underlying parameter):          {true_R0:.3f}")
print()
print(f"R_0 estimate using naive lambda_hat (assumes S*=N):    {R0_lambda_naive:.3f}  [biased ~30%]")
print(f"R_0 estimate using corrected lambda_hat (knows S*):    {R0_lambda_correct:.3f}  [unbiased]")
print(f"R_0 estimate using alpha_hat (no S* needed):           {R0_alpha:.3f}  [unbiased]")

bias_naive = abs(R0_lambda_naive - true_R0) / true_R0
print(f"\nBias of naive lambda_hat: {bias_naive*100:.1f}%")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Panel 1: outbreak trajectory with prior immunity
ax = axes[0]
ax.plot(t, S, color=BOOK_COLORS['susceptible'], lw=1.5, label='S(t)')
ax.plot(t, I, color=BOOK_COLORS['infectious'], lw=1.5, label='I(t)')
ax.axhline(1.0 - prior_immunity_fraction, ls='--', color=BOOK_COLORS['neutral'], alpha=0.5, label=r'$S_0/N$ (true)')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Fraction')
ax.set_title('Seasonal influenza: 30% prior immunity at start')
ax.legend()
ax.set_xlim(0, 100)

# Panel 2: estimator comparison
ax = axes[1]
estimates = [R0_lambda_naive, R0_lambda_correct, R0_alpha, true_R0]
names = [r'$\hat\lambda$ (naive)', r'$\hat\lambda$ (correct $S^*$)', r'$\hat\alpha$', 'True']
colors = [BOOK_COLORS['lambda_hat'], BOOK_COLORS['negative'], BOOK_COLORS['alpha_hat'], BOOK_COLORS['neutral']]
bars = ax.bar(names, estimates, color=colors, alpha=0.85)
ax.axhline(true_R0, ls='--', color=BOOK_COLORS['neutral'], alpha=0.5)
ax.set_ylabel(r'$\hat{R}_0$')
ax.set_title('Three R_0 estimates compared')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()


## Lesson

When prior immunity is present and $S^*$ is uncertain:
- **Naive $\hat\lambda$ assuming $S^* = N$** is systematically biased downward by ~30% (one for every percent of prior immunity)
- **Sophisticated $\hat\lambda$ with correct $S^*$** is unbiased — but requires accurate prior-immunity data
- **$\hat\alpha$** is unbiased without needing prior-immunity data

This is the structural-immunity advantage in a real-world setting. Chapter 18 (dengue) and Chapter 19 (HIV) make analogous comparisons in the actual case studies.